### What i want to discover first:
- Total cost of importing from each country
- Total cost of importing from each supplier (if possible)

Starting from the point that the imported quantity or the Dollar/Kilo price of a product does not affect any data about another product, let us first discover the price of one commodity and then of the other.

#### Sadly i found 3 free suppliers (DataWeb, ImportYeti and Volza), i need to merge them.
I have to take to account that both ImportYeti and Volza take data from the same source

## 1. Setup
Chrome discovery plus the Python 3.14 `distutils` shim that `undetected_chromedriver` still expects. `version_main` is pinned to the installed Chrome major, otherwise the driver download guesses wrong.

In [98]:
import sys, os, re, glob, time, shutil, subprocess, warnings

warnings.filterwarnings('ignore')


class LooseVersion:

    def __init__(self, vstring="0"):
        self.vstring = vstring
        self.version = self._parse(vstring)

    def _parse(self, vstring):
        try:
            return [int(part) for part in vstring.split('.')]
        except ValueError:
            return [0]

    def __lt__(self, other): return self.version < other.version
    def __le__(self, other): return self.version <= other.version
    def __gt__(self, other): return self.version > other.version
    def __ge__(self, other): return self.version >= other.version
    def __eq__(self, other): return self.version == other.version
    def __repr__(self): return f"LooseVersion('{self.vstring}')"


if sys.version_info >= (3, 12) and 'distutils.version' not in sys.modules:
    import types
    _distutils_version = types.ModuleType('distutils.version')
    _distutils_version.LooseVersion = LooseVersion
    _distutils = types.ModuleType('distutils')
    _distutils.version = _distutils_version
    sys.modules['distutils.version'] = _distutils_version
    sys.modules['distutils'] = _distutils


def find_chrome_binary():
    for name in ('google-chrome', 'google-chrome-stable', 'chromium', 'chromium-browser', 'chrome'):
        path = shutil.which(name)
        if path:
            return path
    for path in ('/usr/bin/google-chrome', '/usr/bin/chromium', '/snap/bin/chromium'):
        if os.path.exists(path):
            return path
    return None


def get_chrome_major_version(chrome_binary):
    try:
        out = subprocess.check_output([chrome_binary, '--version'], text=True)
        match = re.search(r'(\d+)\.', out)
        return int(match.group(1)) if match else None
    except Exception:
        return None


CHROME_BINARY = find_chrome_binary()
CHROME_MAJOR = get_chrome_major_version(CHROME_BINARY) if CHROME_BINARY else None
print(f"Chrome binary: {CHROME_BINARY}")
print(f"Chrome major version: {CHROME_MAJOR}")


Chrome binary: /usr/bin/google-chrome
Chrome major version: 151


In [99]:
# Upgrade packages for better Python 3.14 compatibility
import subprocess
import sys

print("Verifying/upgrading packages for Python 3.14 compatibility...")

packages_to_upgrade = [
    'undetected-chromedriver',
    'selenium',
]

for package in packages_to_upgrade:
    try:
        # Correct pip syntax: --upgrade is a separate argument
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            "-q", "--upgrade", package
        ])
        print(f"✓ {package} is up to date")
    except Exception as e:
        print(f"⚠ {package}: {str(e)[:80]}")

print("\n✓ Setup complete! Ready for web scraping")

Verifying/upgrading packages for Python 3.14 compatibility...
✓ undetected-chromedriver is up to date
✓ selenium is up to date

✓ Setup complete! Ready for web scraping


## 2. Browser session
The profile is cloned so a second Chrome inherits the logged-in session; locks and caches are skipped because Chrome refuses to start on a user-data-dir holding another instance's `SingletonLock`.

In [100]:
import json as _json

CHROME_BASE_DIR = os.path.expanduser("~/.config/google-chrome")
CHROME_PROFILE_CLONE_DIR = os.path.join(os.getcwd(), ".chrome_profile_clone")

CLONE_IGNORE = shutil.ignore_patterns(
    "SingletonLock", "SingletonCookie", "SingletonSocket", "lockfile",
    "Cache", "Code Cache", "GPUCache", "ShaderCache", "GrShaderCache",
    "Crash Reports", "Crashpad",
)

STEALTH_JS = """
delete Object.getPrototypeOf(navigator).webdriver;
window.chrome = window.chrome || { runtime: {} };
Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
const _getParameter = WebGLRenderingContext.prototype.getParameter;
WebGLRenderingContext.prototype.getParameter = function(parameter) {
    if (parameter === 37445) return 'Intel Inc.';
    if (parameter === 37446) return 'Intel Iris OpenGL Engine';
    return _getParameter.apply(this, arguments);
};
const _permissionsQuery = navigator.permissions.query;
navigator.permissions.query = (parameters) => (
    parameters.name === 'notifications'
        ? Promise.resolve({state: Notification.permission})
        : _permissionsQuery(parameters)
);
"""

CHROME_PROFILE = "Default"


def list_chrome_profiles():
    local_state_path = os.path.join(CHROME_BASE_DIR, "Local State")
    if not os.path.exists(local_state_path):
        print(f"No 'Local State' at {local_state_path} - open Chrome, log in, then close it fully.")
        return []

    with open(local_state_path, encoding="utf-8") as f:
        info_cache = _json.load(f).get("profile", {}).get("info_cache", {})

    profiles = [(folder, info.get("user_name") or info.get("name") or "(no email)")
                for folder, info in info_cache.items()]
    for folder, name in profiles:
        print(f"  {folder!r} -> {name}")
    return profiles


def clone_chrome_profile():
    if not os.path.isdir(CHROME_BASE_DIR):
        raise FileNotFoundError(f"{CHROME_BASE_DIR} not found. Open Chrome, log in, then close it fully.")
    if not os.path.isdir(os.path.join(CHROME_BASE_DIR, CHROME_PROFILE)):
        raise FileNotFoundError(
            f"Profile {CHROME_PROFILE!r} not found under {CHROME_BASE_DIR}. "
            "Run list_chrome_profiles() to see what exists."
        )

    if os.path.exists(CHROME_PROFILE_CLONE_DIR):
        shutil.rmtree(CHROME_PROFILE_CLONE_DIR)
    print(f"Cloning {CHROME_BASE_DIR} -> {CHROME_PROFILE_CLONE_DIR} ...")
    shutil.copytree(CHROME_BASE_DIR, CHROME_PROFILE_CLONE_DIR, ignore=CLONE_IGNORE)
    print("Clone complete.")
    return CHROME_PROFILE_CLONE_DIR


def create_stealth_driver(user_data_dir):
    import undetected_chromedriver as uc
    from selenium.webdriver.chrome.options import Options

    if not CHROME_BINARY:
        raise RuntimeError("No Chrome/Chromium binary found on this machine")

    options = Options()
    options.binary_location = str(CHROME_BINARY)
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-extensions')
    options.add_argument('--disable-plugins')
    options.add_argument(f'--profile-directory={CHROME_PROFILE}')
    options.set_capability('goog:loggingPrefs', {'browser': 'ALL', 'performance': 'ALL'})

    print(f"Launching Chrome (version_main={CHROME_MAJOR}, profile={user_data_dir})")
    driver = uc.Chrome(
        options=options,
        browser_executable_path=str(CHROME_BINARY),
        version_main=CHROME_MAJOR,
        user_data_dir=user_data_dir,
    )
    try:
        driver.execute_cdp_cmd('Network.enable', {})
        driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {'source': STEALTH_JS})
    except Exception as exc:
        print(f"CDP setup partially failed: {exc}")
    print("Driver ready.")
    return driver


## 3. Volza table helpers
Expanding a row reveals a `table.more-table` with all 31 fields as label/value pairs - that is the scrape target, not the wide collapsed table.

Two gotchas the code works around: the page renders two `.ant-table-body` tables and only one is inside the viewport (the other sits at `left ~= -1146`, where expanding works but is invisible), and rows must be expanded one click per pass because React re-renders after each one.

In [101]:
import pandas as pd

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException, TimeoutException, NoSuchElementException,
)

VOLZA_HOME = "https://app.volza.com/home"
LOGIN_MARKER = "ob-pending-card-title"
EXPAND_SELECTOR = "button.ant-table-row-expand-icon.ant-table-row-expand-icon-collapsed"


def session():
    list_chrome_profiles()
    clone_path = clone_chrome_profile()
    driver = create_stealth_driver(clone_path)
    driver.set_page_load_timeout(30)
    driver.get(VOLZA_HOME)
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.CLASS_NAME, LOGIN_MARKER))
    )
    print(f"Logged-in session confirmed ({LOGIN_MARKER} present)")
    return driver


def on_screen_table(driver):
    infos = driver.execute_script(
        """
        var out = [];
        document.querySelectorAll('.ant-table-body table').forEach(function(t, i) {
            var r = t.getBoundingClientRect();
            out.push({index: i, onScreen: (r.left >= 0 && r.left < window.innerWidth)});
        });
        return out;
        """
    )
    tables = driver.find_elements(By.CSS_SELECTOR, ".ant-table-body table")
    for info in infos:
        if info['onScreen'] and info['index'] < len(tables):
            return tables[info['index']]
    return tables[0] if tables else None


def first_row_key(driver):
    table = on_screen_table(driver)
    if table is None:
        return None
    return driver.execute_script(
        "var r = arguments[0].querySelector('tr[data-row-key]');"
        "return r ? r.getAttribute('data-row-key') : null;",
        table,
    )


def goto_workspace(driver, url):
    print(f"  navigating to {url}")
    driver.get(url)
    try:
        WebDriverWait(driver, 40).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".ant-table-body table"))
        )
    except TimeoutException:
        print("  no .ant-table-body table appeared within timeout")
        return False
    time.sleep(2)
    table = on_screen_table(driver)
    n_expand = len(table.find_elements(By.CSS_SELECTOR, EXPAND_SELECTOR)) if table is not None else 0
    print(f"  table ready; first row key={first_row_key(driver)}, collapsed expand buttons={n_expand}")
    return True


def expand_all_rows(driver):
    max_clicks = 300
    settle = 0.15
    clicked = 0
    while clicked < max_clicks:
        table = on_screen_table(driver)
        if table is None:
            break
        try:
            buttons = table.find_elements(By.CSS_SELECTOR, EXPAND_SELECTOR)
        except StaleElementReferenceException:
            continue
        if not buttons:
            break
        before = len(buttons)
        try:
            driver.execute_script("arguments[0].click();", buttons[0])
        except StaleElementReferenceException:
            continue
        clicked += 1
        time.sleep(settle)
        table = on_screen_table(driver)
        after = len(table.find_elements(By.CSS_SELECTOR, EXPAND_SELECTOR)) if table is not None else 0
        if after >= before:
            print(f"    click {clicked} left the collapsed count at {before} -> {after}; stopping early")
            break
    return clicked


def scrape_expanded(driver):
    table = on_screen_table(driver)
    if table is None:
        return []
    return driver.execute_script(
        r"""
        var out = [];
        arguments[0].querySelectorAll('tr.ant-table-expanded-row').forEach(function(tr) {
            var rec = {};
            var prev = tr.previousElementSibling;
            rec['_row_key'] = prev ? prev.getAttribute('data-row-key') : null;
            tr.querySelectorAll('table.more-table tr').forEach(function(r) {
                var tds = r.querySelectorAll('td');
                if (tds.length >= 2) {
                    var key = (tds[0].innerText || '').trim().replace(/:\s*$/, '');
                    if (key) rec[key] = (tds[1].innerText || '').trim();
                }
            });
            out.push(rec);
        });
        return out;
        """,
        table,
    )


def click_pagination(driver, xpath, li_index):
    timeout = 25
    before_key = first_row_key(driver)
    target = None
    try:
        target = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, xpath))
        )
        print(f"    found via absolute XPath: text={target.text!r}")
    except (TimeoutException, NoSuchElementException):
        items = driver.find_elements(By.CSS_SELECTOR, "ul.ant-pagination > li")
        print(f"    absolute XPath failed; ul.ant-pagination has {len(items)} <li>")
        if len(items) >= li_index:
            target = items[li_index - 1]
            print(f"    falling back to li[{li_index}]: text={target.text!r}")
    if target is None:
        print("    no pagination element found")
        return False

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", target)

    deadline = time.time() + timeout
    while time.time() < deadline:
        time.sleep(0.5)
        now_key = first_row_key(driver)
        if now_key and now_key != before_key:
            print(f"    page changed: first row key {before_key} -> {now_key}")
            return True
    print(f"    first row key never changed from {before_key} within {timeout}s")
    return False


def scrape_workspace(driver, url, page_targets=(), label_prefix="page"):
    if not goto_workspace(driver, url):
        return []

    records = []
    steps = [(f"{label_prefix}_1", None, None)]
    steps += [(f"{label_prefix}_li{li}", xpath, li) for xpath, li in page_targets]

    for label, xpath, li_index in steps:
        print(f"\n=== {label} ===")
        if xpath is not None:
            if not click_pagination(driver, xpath, li_index):
                print(f"  skipping {label}: could not navigate")
                continue
            time.sleep(2)

        n_clicked = expand_all_rows(driver)
        page_records = scrape_expanded(driver)
        for rec in page_records:
            rec['_page'] = label
            rec['_source_url'] = url
        records.extend(page_records)
        print(f"  expanded {n_clicked} rows, scraped {len(page_records)} records")
    return records


def report_and_save(records, out_csv):
    df = pd.DataFrame(records)
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    if not df.empty:
        print(f"Records per page:\n{df['_page'].value_counts()}")
        print(f"Duplicate _row_key values: {df['_row_key'].duplicated().sum()}")
        print("\nFirst rows:")
        print(df.head(3).to_string())
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {len(df)} rows to {out_csv}")
    return df


## 3b. ImportYeti supplier table


In [102]:
YETI_TBODY_XPATH = "/html/body/div[2]/main/div/div/div[4]/div[2]/section/div[2]/div[1]/table/tbody"
EXPAND_SUPPLYERS_XPATH = "/html/body/div[2]/main/div/div/div[4]/div[2]/section/div[2]/div[2]/div/span"
SHIPMENTS_COL = "shipments (01/2015 - 08/2026)"


def yeti_row_count(driver):
    return driver.execute_script(
        "var t = document.evaluate(arguments[0], document, null, 9, null).singleNodeValue;"
        "return t ? t.querySelectorAll(':scope > tr').length : 0;",
        YETI_TBODY_XPATH,
    )


def expand_yeti_suppliers(driver):
    before = yeti_row_count(driver)
    try:
        button = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, EXPAND_SUPPLYERS_XPATH))
        )
    except TimeoutException:
        print(f"  no expand control found; rows stay at {before}")
        return before

    print(f"  expand control text={button.text[:80]!r}, rows before={before}")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", button)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", button)

    grown = before
    deadline = time.time() + 30
    while time.time() < deadline:
        time.sleep(0.5)
        now = yeti_row_count(driver)
        if now > grown:
            grown = now
            continue
        if grown > before:
            print(f"  expanded: rows {before} -> {grown}")
            return grown
    print(f"  row count never grew from {before} within 30s")
    return before


def scrape_yeti_table(driver):
    tbody = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.XPATH, YETI_TBODY_XPATH))
    )
    rows = driver.execute_script(
        """
        var out = [];
        arguments[0].querySelectorAll(':scope > tr').forEach(function(tr) {
            var tds = tr.querySelectorAll(':scope > td');
            if (!tds.length) return;
            var anchors = tds[0].querySelectorAll(':scope > div > div:nth-of-type(1) > a');
            var texts = Array.prototype.map.call(anchors, function(a) {
                return (a.innerText || '').trim();
            });
            var city = texts[1] || '';
            var country = texts[2] || '';
            if (!country) {
                country = city;
                city = '';
            }
            var span = tds.length >= 3
                ? tds[2].querySelector(':scope > div:nth-of-type(1) > span')
                : null;
            out.push({
                name: texts[0] || '',
                city: city,
                country: country,
                shipments: span ? (span.innerText || '').trim() : '',
                n_anchors: texts.length
            });
        });
        return out;
        """,
        tbody,
    )
    kept = [r for r in rows if r['name']]
    print(f"  tbody rows={len(rows)}, with name={len(kept)}")
    return kept


def save_yeti(rows, out_csv):
    df = pd.DataFrame(rows)
    if df.empty:
        print("no rows scraped")
        return df
    print(f"\nanchors per row:\n{df['n_anchors'].value_counts().sort_index()}")
    print(f"missing city={(df['city'] == '').sum()} "
          f"country={(df['country'] == '').sum()} "
          f"shipments={(df['shipments'] == '').sum()}")
    df = df.drop(columns='n_anchors').rename(columns={'shipments': SHIPMENTS_COL})
    df = df[['name', 'city', 'country', SHIPMENTS_COL]]
    print(f"\n{df.head(10).to_string()}")
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {len(df)} rows to {out_csv}")
    return df


## 4. Run

In [103]:
driver = session()


  'Default' -> newtonepv@gmail.com
Cloning /home/newton/.config/google-chrome -> /home/newton/Documents/GitHub/Suppliers/.chrome_profile_clone ...
Clone complete.
Launching Chrome (version_main=151, profile=/home/newton/Documents/GitHub/Suppliers/.chrome_profile_clone)
Driver ready.
Logged-in session confirmed (ob-pending-card-title present)


In [86]:
"""# Workspace 24846649 / Shipments: initial page plus the two pagination steps.
URL_SHIPMENTS = "https://app.volza.com/workspace/search/24846649#Shipments"

# Absolute XPaths for the two pagination targets, each with its <li> index as
# the fallback used when the XPath stops resolving.
SHIPMENT_PAGE_TARGETS = [
    ("/html/body/div[1]/div/div/div[3]/article/div/div/div/div/div[2]/div/div/div/div[2]/div/div[2]"
     "/div[2]/div/div[2]/div[2]/div[2]/div/ul/li[4]", 4),
    ("/html/body/div[1]/div/div/div[3]/article/div/div/div/div/div[2]/div/div/div/div[2]/div/div[2]"
     "/div[2]/div/div[2]/div[2]/div[2]/div/ul/li[5]", 5),
]

df_shipments = report_and_save(
    scrape_workspace(driver, URL_SHIPMENTS, SHIPMENT_PAGE_TARGETS),
    os.path.join(os.getcwd(), "volza_shipments.csv"),
)
"""

'# Workspace 24846649 / Shipments: initial page plus the two pagination steps.\nURL_SHIPMENTS = "https://app.volza.com/workspace/search/24846649#Shipments"\n\n# Absolute XPaths for the two pagination targets, each with its <li> index as\n# the fallback used when the XPath stops resolving.\nSHIPMENT_PAGE_TARGETS = [\n    ("/html/body/div[1]/div/div/div[3]/article/div/div/div/div/div[2]/div/div/div/div[2]/div/div[2]"\n     "/div[2]/div/div[2]/div[2]/div[2]/div/ul/li[4]", 4),\n    ("/html/body/div[1]/div/div/div[3]/article/div/div/div/div/div[2]/div/div/div/div[2]/div/div[2]"\n     "/div[2]/div/div[2]/div[2]/div[2]/div/ul/li[5]", 5),\n]\n\ndf_shipments = report_and_save(\n    scrape_workspace(driver, URL_SHIPMENTS, SHIPMENT_PAGE_TARGETS),\n    os.path.join(os.getcwd(), "volza_shipments.csv"),\n)\n'

In [87]:
"""# Workspace 24846653 / Partial-View. page_targets=() on purpose: this view has
# no usable pagination, so only the rows already loaded get expanded and read.
URL_PARTIAL = "https://app.volza.com/workspace/search/24846653#Partial-View"

df_partial = report_and_save(
    scrape_workspace(driver, URL_PARTIAL, page_targets=(), label_prefix="partial"),
    os.path.join(os.getcwd(), "volza_partial_view.csv"),
)
"""

'# Workspace 24846653 / Partial-View. page_targets=() on purpose: this view has\n# no usable pagination, so only the rows already loaded get expanded and read.\nURL_PARTIAL = "https://app.volza.com/workspace/search/24846653#Partial-View"\n\ndf_partial = report_and_save(\n    scrape_workspace(driver, URL_PARTIAL, page_targets=(), label_prefix="partial"),\n    os.path.join(os.getcwd(), "volza_partial_view.csv"),\n)\n'

## 5. ImportYeti

In [90]:

COUNTRIES_YETI_PATH = ["https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/brazil", "https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/canada", "https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/mexico", "https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/india"]
SUPPLYER_XPATH = "/html/body/div[2]/main/div/div/div[4]/div[2]/div/div[2]/div/div"
EXPAND_SUPPLYERS_XPATH = "/html/body/div[2]/main/div/div/div[4]/div[2]/section/div[2]/div[2]/div/span"
yeti_rows = []
for country_path in COUNTRIES_YETI_PATH:
    driver.get(country_path)
    print(f"loaded: {driver.current_url}")
    target = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.XPATH, SUPPLYER_XPATH))
    )
    print(f"target text before click: {target.text[:200]!r}")
    
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", target)
    time.sleep(0.5)

    target = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.XPATH, SUPPLYER_XPATH))
    )
    print(f"target text before click: {target.text[:200]!r}")
    
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", target)
    time.sleep(0.5)
    expand_yeti_suppliers(driver)
    yeti_rows.extend(scrape_yeti_table(driver))

print(f"url after click: {driver.current_url}")

df_yeti = save_yeti(yeti_rows, os.path.join(os.getcwd(), "importyeti_suppliers.csv"))


loaded: https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/brazil
target text before click: 'SUPPLIERS'
target text before click: 'SUPPLIERS'
  expand control text='Show More', rows before=7
  expanded: rows 7 -> 19
  tbody rows=19, with name=19
loaded: https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/canada
target text before click: 'SUPPLIERS'
target text before click: 'SUPPLIERS'
  expand control text='Show More', rows before=7
  expanded: rows 7 -> 19
  tbody rows=19, with name=19
loaded: https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/mexico
target text before click: 'SUPPLIERS'
target text before click: 'SUPPLIERS'
  expand control text='Show More', rows before=7
  expanded: rows 7 -> 19
  tbody rows=19, with name=19
loaded: https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/india
target text before click: 'SUPPLIERS'
target text before click: 'SUPP

## 6. Debug probe: can we read the hover tooltip?


In [ ]:
from selenium.webdriver.common.action_chains import ActionChains

PROBE_URL = "https://www.importyeti.com/hs-codes/7306-other-tubes-pipes-and-hollow-profiles-for/brazil"
PROBE_SUPPLYER_XPATH = "/html/body/div[2]/main/div/div/div[4]/div[2]/div/div[2]/div/div"
PROBE_SVG_CSS = "table tbody tr td svg.recharts-surface"
PROBE_SAMPLES = 30

PROBE_READ_JS = r"""
var svg = arguments[0];
var wrap = svg.closest('.recharts-wrapper') || svg.parentElement;
var tip = wrap ? wrap.querySelector('.recharts-tooltip-wrapper') : null;
var cursor = svg.querySelector('.recharts-tooltip-cursor');
return {
    mounted: !!tip,
    visibility: tip ? (tip.style.visibility || '') : null,
    lines: tip ? Array.prototype.map.call(tip.querySelectorAll('p'), function (p) {
        return (p.innerText || p.textContent || '').trim();
    }) : [],
    cursorX: cursor ? parseFloat(cursor.getAttribute('x')) : null,
    cursorW: cursor ? parseFloat(cursor.getAttribute('width')) : null
};
"""

driver.get(PROBE_URL)
print(f"loaded: {driver.current_url}")

target = WebDriverWait(driver, 30).until(
    EC.presence_of_element_located((By.XPATH, PROBE_SUPPLYER_XPATH))
)
print(f"suppliers control text: {target.text[:120]!r}")
driver.execute_script("arguments[0].scrollIntoView({block:'center'});", target)
time.sleep(0.5)
driver.execute_script("arguments[0].click();", target)
time.sleep(2)

charts = driver.find_elements(By.CSS_SELECTOR, PROBE_SVG_CSS)
print(f"recharts surfaces on page: {len(charts)}")

if not charts:
    print("no chart rendered after clicking suppliers")
else:
    svg = charts[0]
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", svg)
    time.sleep(0.5)

    rect = driver.execute_script(
        "var r = arguments[0].getBoundingClientRect();"
        "return {w: r.width, h: r.height, left: r.left, top: r.top};",
        svg,
    )
    print(f"svg rect: {rect}")
    print(f"state before any hover: {driver.execute_script(PROBE_READ_JS, svg)}")

    seen = []
    for i in range(PROBE_SAMPLES):
        frac = (i + 0.5) / PROBE_SAMPLES
        x_in_svg = rect['w'] * frac
        offset_x = int(round(x_in_svg - rect['w'] / 2))
        ActionChains(driver).move_to_element_with_offset(svg, offset_x, 0).perform()
        time.sleep(0.2)
        info = driver.execute_script(PROBE_READ_JS, svg)
        print(f"  x={x_in_svg:6.1f}  mounted={info['mounted']}  vis={info['visibility']!r}  "
              f"cursorX={info['cursorX']}  cursorW={info['cursorW']}  lines={info['lines']}")
        if info['lines']:
            seen.append(tuple(info['lines']))

    unique = list(dict.fromkeys(seen))
    print(f"\nhovers that produced a tooltip: {len(seen)}/{PROBE_SAMPLES}")
    print(f"distinct tooltips: {len(unique)}")
    for lines in unique:
        print(f"  {lines}")
